# 05b — Number Probe
Auditoría sobre el manejo de números (`answer_format == "number"`).
Se procesan 2094 ítems para evaluar el comportamiento del parser vs el modelo.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent

for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

from focus.data.data_models import Response, Reference, save_items, load_responses
from frame.config import BaselineConfig
from frame.data import load_frame_items, FrameProvider
from frame.engine import QwenFrameEngine
from focus.enums import Track
from focus.evaluation.evaluator import Evaluator
from focus.evaluation.judges import TransformersJudge

from number_probe import probe_frame, probe_metrics

cfg = BaselineConfig(
    model_path=Path("/workspace/repo/experiments/02-lora-sft/runs/02_lora_sft_v1/merged/checkpoint-1720"),
    out_dir=EXP_DIR / "runs" / "05b_number_probe",
    seed=42
)
cfg.out_dir.mkdir(parents=True, exist_ok=True)

FORCE_PATH_B = False


In [ ]:
# T0: Decidir Camino A o B
run02_dir = REPO / "experiments" / "02-lora-sft" / "runs" / "02_lora_sft_v1" / "eval_best"
preds_path = run02_dir / "predictions.json"
refs_path = run02_dir / "references.json"
results_path = run02_dir / "results.csv"

path_taken = None
if not FORCE_PATH_B:
    if preds_path.exists() and refs_path.exists() and results_path.exists():
        print("T0: Found predictions, references and results.csv for rung 02. Validating acc_OOD...")
        
        results_df = pd.read_csv(results_path)
        if "qID" not in results_df.columns or "correctness" not in results_df.columns:
            raise ValueError("El results.csv del rung 02 no contiene las columnas 'qID' o 'correctness' exactas.")
        
        ood_mask = results_df["qID"].str.startswith("heico")
        acc_ood = results_df[ood_mask]["correctness"].mean()
        
        if abs(acc_ood - 0.5918) <= 0.005:
            print(f"T0 OK: acc_OOD = {acc_ood:.4f} matches checkpoint 1720.")
            path_taken = "A"
        else:
            raise RuntimeError(f"T0 FAIL: acc_OOD is {acc_ood:.4f} (expected ~0.5918). Abortando. Pon FORCE_PATH_B=True para re-correr.")
    else:
        raise RuntimeError("T0 FAIL: predictions.json, references.json o results.csv no encontrados. Abortando. Pon FORCE_PATH_B=True para re-correr.")
else:
    print("FORCE_PATH_B is True. Tomando el Camino B explícitamente.")
    path_taken = "B"


In [ ]:
# Camino A o B: obtener responses, references, results_df para los format=='number'
if path_taken == "A":
    responses = load_responses(preds_path)
    eval_items = load_frame_items(cfg, splits=("test",))
    references = [it.reference for it in eval_items]
    
    number_qids = {r.qID for r in references if str(getattr(r, "_format", "")).lower() == "number"}
    num_responses = [r for r in responses if r.qID in number_qids]
    num_references = [r for r in references if r.qID in number_qids]
    
    num_results_df = results_df[results_df["qID"].isin(number_qids)].copy()
else:
    eval_items = load_frame_items(cfg, splits=("test",))
    number_items = [it for it in eval_items if str(getattr(it.reference, "_format", "")).lower() == "number"]
    
    provider = FrameProvider(cfg)
    engine = QwenFrameEngine(cfg)
    engine.load()
    
    num_responses = []
    num_references = [it.reference for it in number_items]
    num_requests = [it.request for it in number_items]
    
    for i, item in enumerate(number_items):
        provider.ensure_reader(item)
        
        t0 = time.perf_counter()
        img = provider.get_frame(item)
        content = engine.predict(img, item.request.question)
        latency = time.perf_counter() - t0
        
        num_responses.append(Response(qID=item.request.qID, content=content, latency=latency))
        
        if (i + 1) % 50 == 0 or i + 1 == len(number_items):
            print(f"Inferred {i+1}/{len(number_items)}")

    # B2: Guardar los items INMEDIATAMENTE después del bucle, antes de descargar el modelo o cerrar provider.
    # En caso de que reviente el teardown o el juez, el dato costoso ya está en disco seguro.
    save_items(num_responses, cfg.out_dir / "predictions.json")
    save_items(num_references, cfg.out_dir / "references.json")
    save_items(num_requests, cfg.out_dir / "requests.json")
            
    engine.unload()
    provider.close()
        
    judge = TransformersJudge(model_name=cfg.judge_model, device=cfg.device)
    evaluator = Evaluator(judges=[judge], seed=cfg.seed)
    num_results_df, _ = evaluator.run(
        requests=num_requests,
        references=num_references,
        responses=num_responses,
        output_dir=None,
        track=Track.FRAME
    )


In [ ]:
# Analisis
probe_df = probe_frame(num_responses, num_references, results_df=num_results_df)
source = "rung02_predictions" if path_taken == "A" else "rerun_number_only"
metrics = probe_metrics(probe_df, source)

# GATES
assert metrics["n"] == 2094, f"G4 FAILED: Expected 2094 items, got {metrics['n']}"
print("OK G4: n == 2094")

assert probe_df["pred_text"].notna().all() and (probe_df["pred_text"].str.strip() != "").all(), "G1 FAILED: pred_text empty"
assert probe_df["true_value"].notna().all(), "G1 FAILED: true_value holds NaNs"
no_parse_rate = metrics["n_no_parseable"] / metrics["n"]
assert no_parse_rate <= 0.05, f"G1 FAILED: Unparseable rate {no_parse_rate:.3%} exceeds 5%"
print(f"OK G1: pred_text/true_value not empty, unparseable rate {no_parse_rate:.3%} (<= 5%)")

assert "n_no_parseable" in metrics, "G2 FAILED: n_no_parseable not in metrics"
print("OK G2: parsing rules adhered and n_no_parseable reported")

assert abs(metrics["mode_rate"] - 0.3520) <= 0.005, f"G3 FAILED: mode_rate {metrics['mode_rate']:.4f} != 0.3520"
assert abs(metrics["acc_overall"] - 0.4317) <= 0.005, f"G3 FAILED: acc_overall {metrics['acc_overall']:.4f} != 0.4317"
print(f"OK G3: Checksums pass (mode_rate={metrics['mode_rate']:.4f}, acc_overall={metrics['acc_overall']:.4f})")

# C9: Ambiguous negation handling
print(f"\nn_ambiguous_negation: {metrics['n_ambiguous_negation']} / {metrics['n']} ({(metrics['n_ambiguous_negation']/metrics['n']):.2%})")
if metrics['n_ambiguous_negation'] / metrics['n'] > 0.05:
    print("\nATENCIÓN: n_ambiguous_negation supera el 5%. La métrica pred_value NO ES FIABLE para spearman_r, pred_mode_share ni pred_entropy.")
    print("El parser sufre sesgo sistemático a 0.0 en negaciones ambiguas. No interpretar estas tres métricas como definitivas.")

probe_df.to_csv(cfg.out_dir / "probe.csv", index=False)
pd.DataFrame([metrics]).to_csv(EXP_DIR / "RESULTS_number_probe.csv", index=False)
print("\nSaved probe.csv and RESULTS_number_probe.csv")

# 20 raw pred_text samples con C3 request
print("\n--- 20 RANDOM PRED_TEXT SAMPLES ---")
sample_df = probe_df.sample(20, random_state=42)
for _, row in sample_df.iterrows():
    print(f"{row['qID']} | pred: '{row['pred_text']}' -> parsed: {row['pred_value']} | true: {row['true_value']} | corr: {row['correctness']}")
